# Analyse des Ventes MarketCorp



## Phase 1 : Audit & Importation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration esthetique
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 8)

# Chargement des donnees
try:
    df = pd.read_csv('marketcorp_sales.csv')
    print("Donnees chargees avec succes.")
    # Audit rapide - only run if df is loaded
    print(f"\nDimensions du dataset : {df.shape}")
    print("\nTypes de donnees :")
    print(df.dtypes)
    print("\n5 premieres lignes :")
    display(df.head())
    print("\nStatistiques descriptives initiales :")
    display(df.describe())
    print("\nValeurs manquantes par colonne :")
    print(df.isnull().sum())
except FileNotFoundError:
    print("Fichier 'marketcorp_sales.csv' non trouve. ")
    df = None


## Phase 2 : Nettoyage Fiable

Dans cette phase, nous traitons les valeurs manquantes et les incoherences de labels detectees lors de l'audit.

In [ ]:
# 1. Item_Weight : Remplacement des NaN par la moyenne
df['Item_Weight'] = df['Item_Weight'].fillna(df['Item_Weight'].mean())

# 2. Outlet_Size : Remplacement des NaN par 'Inconnu'
df['Outlet_Size'] = df['Outlet_Size'].fillna('Inconnu')

# 3. Item_Visibility : Remplacement des 0.0 par la mediane globale
median_visibility = df[df['Item_Visibility'] > 0]['Item_Visibility'].median()
df['Item_Visibility'] = df['Item_Visibility'].replace(0.0, median_visibility)

# 4. Item_Fat_Content : Uniformisation des labels
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
})

# Validation finale
print("Nettoyage termine.")
print("\nControle final des valeurs manquantes :")
print(df.isnull().sum())

## Phase 3 : Exploration Chiffree

In [ ]:
# 1. Statistiques descriptives des colonnes numeriques
print("Statistiques descriptives :")
display(df.describe())

print("\n--- Constats ---")
print("1. Les ventes varient enormement, indiquant une disparite entre produits stars et produits lents.")
print("2. Le MRP moyen est autour de 140, avec un maximum a 266.")
print("3. La visibilite moyenne est faible (~6%), ce qui suggere une marge d'optimisation.")

# 2. Analyse des variables categorielles
print("\nRepartition Outlet_Size :")
print(df['Outlet_Size'].value_counts())
print("\nRepartition Outlet_Type :")
print(df['Outlet_Type'].value_counts())
print("\nRepartition Item_Fat_Content :")
print(df['Item_Fat_Content'].value_counts())

# 3. Comparaison : Petits vs Grands magasins
sales_small = df[df['Outlet_Size'] == 'Small']['Item_Outlet_Sales']
sales_large = df[df['Outlet_Size'].isin(['Medium', 'High'])]['Item_Outlet_Sales']

print(f"\nVente Moyenne (Petits Magasins) : {sales_small.mean():.2f}")
print(f"Vente Moyenne (Moyens/Grands Magasins) : {sales_large.mean():.2f}")

### Hypotheses Metier
1. Le prix (Item_MRP) est le principal moteur de vente.
2. La visibilite en rayon n'est pas optimale.
3. Les magasins de type 'Supermarket Type 3' generent des ventes plus elevees.

## Phase 4 : Analyse Exploratoire (EDA) - Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Dashboard de Performance Ventes MarketCorp', fontsize=20, fontweight='bold', color='#2c3e50')

# 1. Histogramme des Ventes
axes[0, 0].hist(df['Item_Outlet_Sales'], bins=30, color='#3498db', edgecolor='white')
axes[0, 0].set_title('Distribution des Ventes')
axes[0, 0].set_xlabel('Ventes')
axes[0, 0].set_ylabel('Frequence')

# 2. Bar Chart par Outlet_Size
size_counts = df['Outlet_Size'].value_counts()
axes[0, 1].bar(size_counts.index, size_counts.values, color=['#e74c3c', '#2ecc71', '#f1c40f', '#9b59b6'])
axes[0, 1].set_title('Repartition par Taille de Magasin')
axes[0, 1].set_ylabel('Nombre de Magasins')

# 3. Scatter Plot Item_MRP vs Item_Outlet_Sales
axes[1, 0].scatter(df['Item_MRP'], df['Item_Outlet_Sales'], alpha=0.5, color='#e67e22', s=10)
axes[1, 0].set_title('Prix (MRP) vs Ventes')
axes[1, 0].set_xlabel('MRP')
axes[1, 0].set_ylabel('Ventes')

# 4. Boxplot des Ventes
axes[1, 1].boxplot(df['Item_Outlet_Sales'], vert=False, patch_artist=True,
                  boxprops=dict(facecolor='#95a5a6', color='#2c3e50'),
                  medianprops=dict(color='red'))
axes[1, 1].set_title('Analyse de la dispersion')
axes[1, 1].set_xlabel('Ventes')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Phase 5 : Storytelling & Recommandations

### Synthese pour la Direction Regionale
L'analyse des donnees MarketCorp revele une structure de vente solide mais perfectible. Nous observons une forte correlation entre le prix unitaire (MRP) et le volume de ventes total. Cependant, une part importante des magasins presente des performances en deça des standards. La gestion des stocks et la visibilite des produits semblent etre les leviers majeurs.

### Reponses aux questions strategiques
1.  **Produits performants :** Produits avec un MRP > 200.
2.  **Magasins en difficulte :** Grocery Stores et petits magasins.
3.  **Priorites d'action :** Standardiser l'inventaire et booster la visibilite.

### Recommandations Concretes
1.  Optimiser le Merchandising pour les produits premium.
2.  Dupliquer le modele Type 3.
3.  Mettre en place un systeme de saisie obligatoire pour Outlet_Size.

### Limites de l'Analyse
- Absence de donnees temporelles.
- Biais potentiel de la categorie 'Inconnu'.

### Plan d'action - 30 Jours
- J1-J10 : Audit physique.
- J11-J20 : Reorganisation des rayons.
- J21-J30 : Lancement d'une promotion.

---
## Bonus

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Preparation des donnees
X = df.select_dtypes(include=[np.number]).drop(columns=['Item_Outlet_Sales', 'Outlet_Establishment_Year'])
y = df['Item_Outlet_Sales']
X['Item_Fat_Content_Encoded'] = df['Item_Fat_Content'].map({'Low Fat': 0, 'Regular': 1})

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Entrainement
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Evaluation
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")

print("\n--- Test sur donnees invisibles ---")
try:
    unseen_df = pd.read_csv('marketcorp_unseen_data.csv')
    print("Fichier 'marketcorp_unseen_data.csv' charge.")
except:
    print("Fichier invisible non charge.")